# 수험번호를 반드시 입력하세요!

**문제를 시작하기 전에, 바로 아래 셀의 `student_id`에 본인의 수험번호와 본인의 LLM API를 입력하고 실행하세요.**

예: `student_id = '001234'` — 앞자리 0도 포함하여 따옴표 안에 입력하세요.


In [ ]:
# 본인의 수험번호를 따옴표 안에 입력하세요. 예: '001234'
student_id = ''
if not student_id or not student_id.isascii() or not student_id.isdigit():
    raise ValueError('수험번호를 숫자로만 입력하세요. 앞자리 0도 그대로 적습니다.')
print('수험번호:', student_id)


In [ ]:
# 본인의 API 키를 따옴표 안에 입력하고 실행하세요. 이후 모델 호출에 사용됩니다.
USER_API_KEY = 'dummy'
if not isinstance(USER_API_KEY, str) or not USER_API_KEY.strip():
    raise ValueError('USER_API_KEY에 본인의 API 키를 입력하세요.')
USER_API_KEY = USER_API_KEY.strip()


## 코드 작성 안내
- 답안 작성과 관련 있는 셀은 앞쪽에 `답안 작성 셀`로 시작하는 마크다운 셀이 있습니다.
- 수정이 불가한 셀에는 `# 수정 금지`로 표시가 되어있습니다. 해당 주석이 있는 셀의 코드는 절대 수정하지 마세요.
- 수정 가능한 셀은 `# 수정 가능`으로 표시가 되어있습니다. 다만 셀 안에서 함수명 등 변경하면 안되는 부분도 있으니, 그런 경우 해당 셀 혹은 문항의 안내를 참고하세요.

# 문제 3. 언어모델 Prompting 기반 지능형 함수 구현

### 배경
본 문제에서는 입력 `x`를 지능적으로 처리하여, 출력 `y`를 뱉어내는 함수 `f()`를 구현하는 문제입니다. (즉, `y = f(x)`) \
기존에 `f()`를 구현하는 방법은 내부를 프로그래밍 언어로 구현하거나, 뉴럴넷으로 함수 `f()`를 구현하고 데이터로 학습시키는 형태로 구현했으나,\
본 문제에서는 한글 설명 (프롬프트)만으로 지능적 함수 `f()`를 구현합니다.

### 문제 지시문
부동산 거래 녹취를 기반으로 자동으로 계약서 폼을 작성해 채워주는 지능적 함수를 "한글"로 구현하세요.

본 문제에서는 전세계약 녹취(가상 시나리오)에서 계약 정보를 추출하여 계약서 폼 작성에 필요한 정보만 요약하여 포맷에 맞춰 출력하는 함수를 구현합니다.\
함수의 구현은 한글 기반 **시스템 프롬프트**를 작성으로 구현합니다.\
녹취에는 조건 변경, 정정, 합의되지 않은 제안 등이 포함될 수 있습니다.

- 수험번호를 입력한 뒤, 답안은 `system_prompts()` 함수 안에서만 작성하세요. 모델 호출, 응답 검증과 파일 저장 코드는 수정하지 마세요.
- **제공된 연결 파일과 호출 코드는 평가 조건이므로 변경할 수 없습니다.** 언어모델 연결 서버·모델 설정을 바꾸거나 별도 클라이언트로 우회 호출하는 등의 어떠한 시도도 금지합니다.
- **만약 제공된 LLM을 다른 목적으로 이용하는 경우, 즉시 부정행위 처리 될 수 있습니다**
- 모든 데이터 샘플에 적용할 단일 프롬프트를 작성하세요. 문항별 정답을 직접 넣거나 함수 안에서 모델을 별도로 호출하지 마세요.

녹취와 계약 내용은 평가용 창작이며, 필요한 정보는 녹취에 있습니다.


## 환경 확인

먼저 아래 셀로 노트북 커널을 확인하세요. 이 폴더의 `.venv\Scripts\python.exe`를 선택해야 합니다.\
필요한 패키지는 `setup.cmd`가 설치합니다. 


In [ ]:
# 수정 금지
# 먼저 현재 커널과 OpenAI SDK를 확인하세요.

import sys
from pathlib import Path

print('Python:', sys.version.split()[0])
print('현재 커널:', sys.executable)
print('작업 폴더:', Path.cwd())
expected_env = Path.cwd() / '.venv'
if Path(sys.prefix).resolve() != expected_env.resolve():
    raise RuntimeError(f"{expected_env / 'Scripts' / 'python.exe'}를 노트북 커널로 선택하세요.")
try:
    import openai
except ImportError as error:
    raise RuntimeError('OpenAI SDK가 없습니다. setup.cmd로 환경을 준비하고 커널을 다시 시작하세요.') from error
print('OpenAI SDK:', openai.__version__)
print('커널 확인 완료. 다음 셀로 진행하세요.')


In [ ]:
# 수정 금지

import csv
from datetime import date
import json
import math
from pathlib import Path
import pickle
import time


## 데이터 예시 및 응답 Schema

노트북과 같은 폴더의 `P3_qa.csv`에 프롬프트 테스트용 예제 2개가 제공됩니다.\
한 행이 한 문제이며, 아래 두 열만 포함합니다. 정답은 공개하지 않습니다.

| 열 | 의미 |
| --- | --- |
| `task` | 수행할 작업 이름 |
| `question` | 작업에 사용할 녹취 전문 |

아래 두 예시는 `P3_qa.csv`의 내용입니다.


In [ ]:
# 수정 금지

def load_questions(path):
    """task, question 두 열이 있는 CSV를 문항 목록으로 읽습니다."""
    with Path(path).open(encoding='utf-8-sig', newline='') as source:
        reader = csv.DictReader(source)
        if reader.fieldnames != ['task', 'question']:
            raise ValueError('CSV는 task, question 두 열이어야 합니다.')
        questions = [
            {'task': row['task'], 'question': row['question'].replace('\\n', '\n')}
            for row in reader
        ]
    if not questions:
        raise ValueError('CSV에 문항이 없습니다.')
    return questions

examples = load_questions('P3_qa.csv')
print('예제 수:', len(examples))


### 예시 1: 전세계약 정보 추출

2027-01-09 · 매물 거실에서 서류를 펴 놓은 자리



이거 끝까지 닫힌 거예요? 손잡이를 내려도 여기서 바람이 좀 오는데. 아, 커튼이 움직이네. 제가 어제 밤에 생각해 보니까 아이 침대를 이쪽에 놓아야 해서 그냥 넘어가기는 어려울 것 같아요. 잠깐만요, 위를 조금 들어서 밀면 돼요. [창문 미는 소리] 이게 처음부터 그런 건 아니고 지난겨울쯤부터 내려앉았어요. 유리까지 갈아야 하는 건 아니라고 했는데 부품을 기다리고 있습니다. 서경 씨, 창호 업체 사진 받은 것부터 보실래요? 틀 자체보다 롤러 문제라고 했어요. 이 부분을 누가 처리할지 정하고 가격 얘기를 하는 게 순서일 것 같습니다. 네. 임서경으로 서류 준비해 왔고요. 남편은 오늘 못 왔는데 집은 제 이름으로 하기로 했어요. 회사에서 주거비 서류를 내라 그래서 제가 당사자로 들어가야 해요. 남편이 보낸 메시지에 적힌 날짜는 아직 이사업체에 물어본 것뿐이고요.



제가 롤러하고 작은방 벽지까지 해드리면 처음 말씀드린 사억 삼천에서 오백은 빼드릴 수 있어요. 지금 가구가 있어서 벽 전체를 확인하지 못하는 건 감안하셔야 하고요. 그럼 사억 이천오백인데, 벽지는 제가 고르고 싶거든요. 애가 냄새에 예민해서요. 창문만 고쳐 주시고 벽지는 제가 하는 것으로 더 내려갈 수 있을까요? 도윤 씨가 받은 견적에는 거실 벽지까지 같이 들어가 있어요. 작은방만 빼는 건지 벽지 작업을 다 빼는 건지부터 맞춰야겠네요. 나중에 서로 다른 견적을 이야기하면 곤란해서요. 강도윤, 여기 소유자 이름 맞고요. 저는 벽지 작업 전부 빼는 쪽이면 사억 천팔백까지 하겠습니다. 대신 창문 롤러는 제가 고치고, 현관 손잡이도 헐거운 건 잡아놓을게요.



그렇게 할게요. 벽지는 저희가 하고 창문하고 손잡이는 고쳐 주시는 걸로요. 싱크대 문도 약간 어긋나긴 하는데 이건 제가 기사 올 때 같이 봐달라고 하겠습니다. [종이 넘김] 이쪽에 앉으세요. 바닥에서는 쓰기 불편하니까요. 이건 제가 초록문공인중개사사무소에서 출력해 온 대장이고, 노기윤 이름 찍힌 명함도 여기 드릴게요. 저번에 현관만 열어드린 직원하고 헷갈리셨다고 해서요. 아, 저는 그분이 오늘 서류도 쓰시는 줄 알았어요. 그러면 노기윤 소장님하고 계속 이야기하면 되는 거죠? 제 이메일 끝에 붙은 옛집 주소는 빼주세요. 아직 자동으로 붙어 나가더라고요.



네. 여기 서울특별시 송파구 올림픽로 사백삼십오, 파크리오아파트인데 지금 앉아 있는 곳이 백십일동 구백이호예요. 아까 주차한 백십동 쪽과는 입구가 달라요. 법정동은 신천동으로 들어갑니다. 엘리베이터에서 오른쪽으로 오신 집 맞아요. 전용은 팔십사 점 칠육이고, 이 줄이 철근콘크리트조네요. 제가 분양받을 때 받은 도면에는 다른 면적도 크게 적혀 있는데 그건 공급면적이에요. 침대 놓을 폭은 아까 재둔 걸로 충분하겠네요. 전용 숫자가 인터넷 사진과 조금 달라서 궁금했어요. 숫자를 잘못 적은 광고가 있었나 봐요. 광고 캡처에 있던 팔십사 점 구칠은 이 집 자료가 아니에요. 같은 단지의 다른 평면을 같이 보내드렸던 것 같습니다. 오늘 가져온 이 호수 자료로 진행할게요.



계약금은 삼천팔백으로 하고 싶어요. 비율로 맞추기보다 그 액수까지는 제가 바로 마련할 수 있어서요. 먼저 보낸 팔백도 안에 넣고요. 나머지를 하루에 다 보내지는 못할 것 같은데 괜찮으실까요? 팔백은 들어온 걸 확인했어요. 그걸 포함해서 삼천팔백이면 받겠습니다. 중간에 한 번 더 받는 돈은 없어도 되고, 나머지를 집 넘기기 전에 받으면 돼요. 그러면 서명은 이번 달 십삼일 오후에 하죠. 십삼일에 일부를 더 보내고, 십오일까지는 계약금을 다 채울게요. 중간 지급 없이 나머지는 마지막에 드리는 것으로 할게요.



그렇게 합시다. 제가 십삼일 오전에는 일이 있어서 오후 네 시쯤 여기 근처 사무실로 가면 됩니다. 십오일에 다 채우는 것도 괜찮고요. 잔금은 이월 이십일일을 원하셨죠? 서경 씨가 처음 종이에 써주신 날짜는 그날인데 은행 예약이 잡혔는지요.



그날은 제가 잘못 생각했어요. 돈이 이십이일에 나옵니다. 지금 제안드리는 건 이십이일이고, 그날 오전에 보내면 좋겠어요. 서명 날짜를 바꾸자는 건 아닙니다. 이십이일에 받는 것으로 하죠. 지금 사는 분 짐은 그날 저녁에 다 나가니까 제가 확인하고 다음 날 집하고 열쇠를 넘길게요. 하루는 제가 정리해야 해서 돈 받는 즉시 들어오시는 건 어렵고요. 괜찮아요. 집 받는 날부터 기간이 시작되는 것으로 하고, 끝은 이천이십구년 이월 이십이일까지로 하죠. 실제 짐은 집 받은 사흘 뒤에 넣을게요. 먼저 도배할 시간이 필요해서요.



그 기간이고 사흘 뒤 이사하는 것으로 받겠습니다. 도배 기사 오실 때 비밀번호는 직접 바꾸셔도 돼요. 대신 작업하다 경보 울리면 어디에 연락하는지는 알려드릴게요. 사다리차가 서는 곳이 놀이터 쪽이면 관리실에 미리 말을 해야 해요. 지금 보고 있는 창으로 가구를 넣을지, 엘리베이터로 올릴지에 따라 신청이 달라져서 이사업체하고 한번 맞춰보세요. 기사님이 이십팔일이 더 싸다고 하긴 했는데 저는 휴가를 이미 정해서요. 그 날짜로 옮기겠다는 건 아닙니다. 스케줄은 방금 말씀드린 대로 갈게요. 아까 벽지 제가 한다고 했으니 천오백까지는 안 될까요?



사억 천오백 말씀하시는 거면 그건 어렵습니다. 창문 고치는 비용은 제가 부담하는 것으로 이미 내려드린 거라서요. 네, 더 말씀 안 드릴게요. [줄자 감기는 소리] 이 현관 안쪽 폭만 다시 재도 될까요? 지난집에서 쓰던 장이 안 들어갈까 봐요. 신발장 문을 열면 여기서 조금 더 줄어들어요. 문 떼고 넣을 수 있는지는 업체에 사진 보내보세요. 사진은 제가 반대편에서 하나 찍어드릴게요.



그 장이 들어갈 때 이 손잡이만 조심하면 됩니다. 바닥에 붙여놓은 표시 테이프는 제가 나중에 떼어놓겠습니다.


### 예시 2: 전세계약 정보 추출

기록 날짜: 2027-03-02

중개사무소 원탁



[A] 여기 이름부터 고쳐야겠는데요. 박하린 아니고 박하민이에요. 끝에 민. 동생이 하린이라 그 사람이 보내준 파일에서 그대로 옮기신 것 같아요. 제가 들어가서 살 거고 제 이름으로 쓰는 겁니다.

[C] 죄송해요, 예약표 이름을 먼저 가져왔네요. 이 종이는 제가 새로 뽑을게요. 하민 씨 이름 들어간 서류는 이쪽에 따로 있고요. 동생분은 방 볼 때 함께 오셨지만 오늘은 세 분 것만 놓고 확인하면 됩니다.

[B] 저도 처음엔 하린 씨로 알고 있었어요. 집 보러 왔을 때 이야기하신 분이 하민 씨군요. 저는 윤서진이고, 아까 주차권 드릴 때 이름이 안 들리셨다고 해서요.

[A] 네, 서진 씨. 저번에 주차장 출구에서 정신이 없었어요. 제가 가지고 온 계약금 영수증에도 이름이 잘못돼 있나요?

[C] 그건 박하민으로 되어 있어요. 보내신 계좌 명의로 확인했거든요. 계약금에 들어갈 돈 오백만원은 맞게 연결돼 있습니다. 프린터가 잠깐 예열 중이라 차 먼저 드세요.

[B] 이 두 종이 주소가 왜 달라요? 하나에는 충장로 백십팔 다시 삼십이고, 하나는 제가 사는 집 주소인데요. 둘 다 소재지라고 표시돼 있어서요.

[C] 왼쪽은 임대차 목적물, 오른쪽은 서진 씨 연락 주소를 옮겨놓은 메모예요. 제가 표 머리글을 붙이지 않아서 혼동되게 보이네요. 계약할 집은 경기도 고양시 덕양구 충장로 118-30이고, 샘터마을2단지아파트입니다.

[A] 이백사동 팔백사호, 맞죠? 제가 앱에 이백오동으로 저장해 놓은 건 다른 집 볼 때예요. 그 집은 이사가 늦어질 수 있다고 해서 제외했던 거고요.

[B] 맞아요. 행신동에 있는 이백사동 팔백사호가 제 집이에요. 도로 번호 뒤의 삼십은 호수가 아니고요. 배달 기사가 가끔 그걸 동 번호로 받아 적더라고요.

[C] 여기 대장에는 철근콘크리트조, 전용 오십구 점 구오로 나옵니다. 아까 인쇄물 맨 위의 칠십몇 제곱미터는 공급면적입니다. 하민 씨가 재신 가구 크기는 평면도에 표시해 둘게요.

[A] 가격은 지난번 삼억에서 백오십 내려주신다는 쪽으로 생각하고 왔어요. 다만 계약금을 삼천까지는 못 맞추고 이천팔백오십으로 하고 싶습니다. 오백 먼저 넣은 것을 포함해서요.

[B] 가격 백오십 낮추는 것과 계약금 그 액수, 둘 다 받겠습니다. 중도금을 육천 정도 주시면 제가 다음 집 계약할 때 편하긴 합니다.

[A] 육천까지는 어렵습니다. 처음엔 사천을 말씀드렸는데 제 적금이 그때 다 안 풀릴 수 있어요. 우선 이천오백까지는 확실합니다.

[B] 그러면 이천오백을 중간에 받고 나머지를 마지막으로 미루죠. 전체 보증금을 더 낮춘다는 뜻으로 이야기한 건 아닙니다.

[A] 네, 중도금만 그렇게 줄이는 것으로 이해했습니다. 계약금은 지금 정한 대로 두고요. 동생이 빌려줄 돈은 아직 확정이 안 돼서 약속에 넣지는 않겠습니다.

[C] [의자 끄는 소리] 새 인쇄물 나왔어요. 여기 이름 확인해 주세요. 아래 사무소 이름은 고운자리공인중개사사무소, 장유경 제 이름으로 찍혀 있습니다. 계약서 서명도 제가 합니다.

[A] 네, 이제 박하민으로 맞네요. 이름 바뀐 종이만 가지고 가고, 앞의 잘못 나온 것은 제가 사진을 안 찍는 게 낫겠어요. 은행에 잘못 제출할 수 있으니까요.

[B] 서명하러 오는 날은 삼월 사일로 하죠. 저는 사일 저녁까지는 시간이 됩니다. 계약금은 그날 전부 받는 것으로 생각하면 되나요?

[A] 사일에 서명하고 나머지 계약금은 육일에 다 보내겠습니다. 오백은 이미 들어갔으니 추가분을 그날 채우는 거예요. 이틀 차이 나는 것까지 받아주시면 좋겠습니다.

[B] 사일에 쓰고 육일에 채우는 것으로 받겠습니다. 중도금은 십구일, 잔금은 사월 팔일이면 제 일정하고 맞습니다.

[A] 네, 삼월 십구일과 사월 팔일로 하겠습니다. 집을 받는 것은 잔금 다음 날로 이야기했던 것 같은데 그건 그대로 가능한가요?

[B] 구일에 청소를 마치고 집하고 열쇠를 드릴게요. 기간도 그날 시작해서 이천이십구년 사월 팔일까지로 합시다. 실제 이사는 그때 바로 하실 건가요?

[A] 열쇠 받은 이틀 뒤에 하겠습니다. 제 원래 집에서 짐 빠지는 날을 한번에 잡아야 해서요.

[B] 그렇게 하세요. 하루 먼저 짐을 조금 넣고 싶으시면 그건 나중에 관리실 시간에 맞춰보시고요.

[A] 아, 적금 내역 보니까 제가 잘못 봤어요. 십구일 전에 풀리는 게 하나 더 있네요. 아까 줄이자고 했던 이천오백에 천오백을 더 낼 수 있습니다. 처음 육천까지 돌아가자는 건 아니고 그만큼만요.

[B] 그러면 지금 중도금에 천오백 더하는 것으로 바꾸죠. 날짜는 십구일 그대로면 됩니다. 잔금에서는 그만큼 빠지면 되고요.

[A] 네. 동생에게 빌릴 돈이 아니라 제 적금이라 그 액수는 약속할 수 있습니다. 추가 계약금을 더 내겠다는 이야기는 아닙니다.

[C] 확인했습니다. 삼월 사일에 오실 때 오늘 받은 이 종이를 가져오시면 고친 부분 비교가 쉬워요. 제가 이전 출력물은 섞이지 않게 따로 넣어두겠습니다.

[B] 주차는 지하 말고 뒤편에 하세요. 문 옆에 공사 자재가 있어서 길이 조금 좁아졌더라고요.

[A] 그럼 걸어서 올게요. 지금 이 커피 컵은 어디에 놓으면 될까요? 서류에 쏟을까 봐 여기 치우고 가겠습니다.


### **지켜야 할 응답 형식 (중요!)**

***사용자가 작성한 시스템 프롬프트는 언어모델이 하나의 녹취 입력를 기반으로, 아래 모든 항목을 포함한 JSON 객체 하나를 반환하도록 유도되어야 합니다.*** \
언어모델로부터 반환된 응답이 JSON형태로 구조화된 반환이 성공하면, 제공된 `call_llm()` 함수 내에서 아래와 같이 파싱된 Python 딕셔너리 형태로 반환하며, 값의 형식은 다음과 같습니다.

| 항목 | Python 값의 형식 | 예시 |
| --- | --- | --- |
| 사람·주소·건물 정보 | `str` | `'가상 인물'`, 건물번호 `'12-3'`, 상세주소 `'103동 804호'` |
| 금액 | 원 단위 `int` | `100000000` |
| 전용면적 | 제곱미터 단위 `int` 또는 `float` | `50.5` |
| 날짜 | `str` | `'2030-01-02'` |
| 확인할 수 없거나 합의되지 않은 값 | `None` | `None` |

Python에서는 값이 없음을 **`None`**으로 나타냅니다. 모델이 출력하는 JSON에서는 **`null`**이며, `json.loads()`가 이를 `None`으로 변환합니다. 저장할 때는 `json.dumps()`가 반대로 변환합니다. 문자열 `"None"`이나 `"null"`을 넣지 마세요.

- 날짜는 **`YYYY-MM-DD`(연 4자리-월 2자리-일 2자리)** 문자열입니다. 예를 들어 2030년 1월 2일은 `'2030-01-02'`입니다.\
`'2030-1-2'`처럼 0을 생략하거나 시각을 붙이지 마세요. `'2030-02-30'`처럼 존재하지 않는 날짜도 허용하지 않습니다.
- 명도일·입주일·임대차 시작일은 각각 확인하고, 미확정 날짜는 Python에서 `None`, JSON에서 `null`로 표시합니다.
- 중도금이 없다고 명시된 경우에는 금액이 `0`, 지급일이 `None`입니다.

아래 `response_schema`는 JSON의 키와 자료형을 정의한 **JSON Schema**입니다.\
여기서 `'null'`은 JSON Schema의 타입 이름입니다. 예를 들어 `['string', 'null']`은 파싱 후 문자열 또는 `None`을 허용한다는 뜻입니다.
프롬프트를 작성할 때 이 딕셔너리를 사용해도 됩니다.

참고: [Python JSON 변환](docs/pages/docs.python.org/3_library_json--c9c50a504fb8.html#encoders-and-decoders), [날짜 검사](docs/pages/docs.python.org/3_library_datetime--30088197847a.html#datetime.date.fromisoformat), [JSON Schema의 null](docs/pages/json-schema.org/understanding-json-schema_reference_null--0e90a1ea0c73.html)


In [ ]:
# 수정 금지

response_schema = {
    'type': 'object',
    'properties': {
        '임대인': {'type': ['string', 'null']},
        '임차인': {'type': ['string', 'null']},
        '중개인': {'type': ['string', 'null']},
        '중개사무소': {'type': ['string', 'null']},
        '시도': {'type': ['string', 'null']},
        '시군구': {'type': ['string', 'null']},
        '읍면': {'type': ['string', 'null']},
        '도로명': {'type': ['string', 'null']},
        '건물번호': {'type': ['string', 'null']},
        '상세주소': {'type': ['string', 'null']},
        '법정동': {'type': ['string', 'null']},
        '건물명': {'type': ['string', 'null']},
        '건물구조': {'type': ['string', 'null']},
        '전용면적_m2': {'type': ['number', 'null'], 'minimum': 0},
        '전세보증금_원': {'type': ['integer', 'null'], 'minimum': 0},
        '계약금_원': {'type': ['integer', 'null'], 'minimum': 0},
        '중도금_원': {'type': ['integer', 'null'], 'minimum': 0},
        '잔금_원': {'type': ['integer', 'null'], 'minimum': 0},
        '계약체결일': {'type': ['string', 'null'], 'format': 'date'},
        '계약금지급일': {'type': ['string', 'null'], 'format': 'date'},
        '중도금지급일': {'type': ['string', 'null'], 'format': 'date'},
        '잔금지급일': {'type': ['string', 'null'], 'format': 'date'},
        '명도일': {'type': ['string', 'null'], 'format': 'date'},
        '입주일': {'type': ['string', 'null'], 'format': 'date'},
        '임대차시작일': {'type': ['string', 'null'], 'format': 'date'},
        '임대차종료일': {'type': ['string', 'null'], 'format': 'date'},
    },
    'additionalProperties': False,
}
response_schema['required'] = list(response_schema['properties'])


# **답안 작성 셀 - 시스템 프롬프트 작성**

`system_prompts()` 함수를 완성하세요. 입력 인자는 없으며, 반환값은 **모든 문항에 공통으로 적용할 시스템 프롬프트 문자열 단 하나**입니다.\
역할, 추출 규칙, 출력 형식을 이 함수 안에서 작성하세요.

### API에 전달되는 메시지

| API 역할 | 전달되는 내용 | 작성 방법 |
| --- | --- | --- |
| `system` | `system_prompts()`의 반환값 | 함수 안에서 공통 지시문 작성 (응시자가 작성해야하는 부분) |
| `user` | `problem['question']` | 제공 코드가 해당 문항의 녹취 원문을 그대로 전달 |

`question`을 별도 프롬프트로 만들거나 시스템 프롬프트에 붙이는 것은 금지됩니다.\
아래 제공된 호출 코드는 시스템 지시문과 녹취를 두 메시지로 구분해 전달합니다.\
각 요청은 독립적이므로 이전 문항의 대화에 의존하지 마세요.

### JSON Schema에 맞춰 응답받기

위 `response_schema`는 참고용으로 정의되었습니다. 제공된 코드에서는 언어모델에 전달되지 않습니다.\
이 스키마를 사용하고 싶다면, `json.dumps(response_schema, ensure_ascii=False)`로 문자열을 만들어 시스템 프롬프트에 넣을 수 있습니다.\
스키마와 함께 **어떤 응답을 원하는지**도 구체화해서 프롬프트를 만드세요. 출력 지시의 예시는 다음과 같습니다.

> 아래 JSON Schema에 맞는 JSON 객체 하나만 출력하세요. 모든 키를 포함하고, 설명이나 코드 블록은 붙이지 마세요.

모델이 반환할 것은 Schema 자체가 아니라 **추출한 값이 채워진 객체**입니다.\
키와 문자열에는 큰따옴표를 쓰고, 미확정 값은 JSON의 `null`로 표현하도록 안내하세요. 숫자·날짜 형식은 앞의 표를 따릅니다.\
이 문제는 API의 Structured Outputs나 JSON mode를 설정하지 않습니다. 프롬프트로 형식을 요청한 뒤, 제공 코드로 검증합니다.

### 공식 문서

- [OpenAI Python SDK 사용법](docs/pages/developers.openai.com/api_reference_python--7362aaaa36b8.html): `OpenAI` 객체와 `chat.completions.create` 호출 예시
- [Chat Completions API](docs/pages/developers.openai.com/api_reference_resources_chat--ef63818ccfcb.html): `messages`, 역할과 응답 구조
- [OpenAI Prompt engineering](docs/pages/developers.openai.com/api_docs_guides_prompt-engineering--fcee37c55ff3.html): 지시문, 문맥과 출력 형식 작성
- [Structured Outputs](docs/pages/developers.openai.com/api_docs_guides_structured-outputs--7c7a6f9a9831.html): API에서 출력 형식을 지정하는 별도 기능. 이 문제에서는 사용하지 않습니다.

문서의 호출 예시는 참고용입니다. 제공된 연결 파일과 호출 코드를 수정하거나 별도 클라이언트로 우회 호출하면 안 됩니다.


# **답안 작성 셀**

In [ ]:
# 수정 가능
# 함수명은 변경하지 마세요.

def system_prompts():
    """모든 문항에 공통으로 적용할 시스템 프롬프트를 반환합니다.

    입력 인자는 없습니다. 녹취 원문은 호출 코드가 question에서 가져와
    별도의 user 메시지로 전달합니다.

    Returns:
        str: 역할, 추출 규칙, 출력 형식을 담은 시스템 지시문입니다.
    """
    system_prompt = '주어진 녹취에서 계약 정보를 JSON으로 정리하세요.'
    return system_prompt

## 응답 검증

### 응답에 붙을 수 있는 형식

아래는 **두 항목만 표시한 축약 예시**입니다. 실제 응답에는 Schema의 26개 키가 모두 필요합니다.

**요청할 형식: JSON 객체만 출력**

```text
{"계약체결일": "2030-01-02", "중도금지급일": null}
```

**Markdown 코드 블록(fence)이 붙은 응답**

````text
```json
{"계약체결일": "2030-01-02", "중도금지급일": null}
```
````

백틱 세 개는 코드 블록의 경계이고, 뒤의 `json`은 언어 표시입니다. 둘 다 JSON 객체의 일부가 아닙니다.

**태그가 붙은 응답**

```text
<json>
{"계약체결일": "2030-01-02", "중도금지급일": null}
</json>
```

`<json>` 같은 태그나 “결과입니다”라는 설명이 붙을 수도 있습니다. 프롬프트에서는 이런 장식 없이 객체만 요청하세요.

### 이 문제의 파싱 방식

제공하는 `parse_response()`는 다음 순서로 처리합니다.

1. `find('{')`와 `rfind('}')`로 **첫 `{`부터 마지막 `}`까지** 잘라냅니다.
2. `json.loads()`로 Python 딕셔너리를 만듭니다. JSON의 `null`은 `None`으로 변환됩니다.
3. 26개 키가 모두 있는지, 자료형·숫자 범위·날짜 형식이 맞는지 검사합니다.

코드 블록이나 태그를 따로 인식하는 것은 아닙니다. JSON 바깥에 다른 중괄호가 없으면, 위의 fence·태그나 `json` 머리말은 잘라낸 범위에서 제외됩니다.\
여러 객체를 연달아 출력하거나 바깥 설명에 `{}`가 있으면 파싱이 실패할 수 있습니다. 작은따옴표, Python의 `None`, 마지막 항목 뒤 쉼표 등 잘못된 JSON 문법도 자동으로 고치지 않습니다.

형식 오류가 나면 프롬프트를 수정하세요. 파싱 성공 여부와 추출 내용의 정확성은 별도로 평가합니다.


In [ ]:
# 수정 금지

def parse_response(text):
    """응답의 첫 {부터 마지막 }까지 읽고, 이 문제의 Schema를 검사합니다."""
    if not isinstance(text, str):
        raise ValueError('모델 응답이 문자열이 아닙니다.')
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end < start:
        raise ValueError('응답에서 JSON 객체를 찾지 못했습니다.')

    answer = json.loads(text[start:end + 1])
    properties = response_schema['properties']
    if not isinstance(answer, dict) or set(answer) != set(properties):
        raise ValueError('JSON의 키가 Schema와 다릅니다. 누락되거나 추가된 키를 확인하세요.')

    type_names = {str: 'string', int: 'integer', float: 'number'}
    for name, rule in properties.items():
        value = answer[name]
        if value is None:
            continue
        value_type = type_names.get(type(value))
        if value_type == 'integer' and 'number' in rule['type']:
            value_type = 'number'
        if value_type not in rule['type']:
            raise ValueError(f'{name}: 자료형이 다릅니다. 필요한 형식: {rule["type"]}')
        if isinstance(value, float) and not math.isfinite(value):
            raise ValueError(f'{name}: 유한한 숫자여야 합니다.')
        if 'minimum' in rule and value < rule['minimum']:
            raise ValueError(f'{name}: 음수는 허용하지 않습니다.')
        if rule.get('format') == 'date':
            message = f'{name}: YYYY-MM-DD 형식의 실제 날짜를 입력하세요. 예: 2030-01-02'
            try:
                parsed_date = date.fromisoformat(value)
            except ValueError as error:
                raise ValueError(message) from error
            if parsed_date.isoformat() != value:
                raise ValueError(message)
    return answer


In [ ]:
# 수정 금지
# 형식 확인용 예시이며, 실제 문항의 정답이 아닙니다. API는 호출하지 않습니다.

example_answer = {name: None for name in response_schema['properties']}
example_answer['임대인'] = '가상 인물'
example_answer['전용면적_m2'] = 50.5
example_answer['전세보증금_원'] = 100000000
example_answer['계약체결일'] = '2030-01-02'
example_response = json.dumps(example_answer, ensure_ascii=False)
parsed = parse_response(example_response)
print('날짜 문자열:', parsed['계약체결일'])
print('Python 값:', parsed['중도금지급일'])
print('JSON 값:', json.dumps(parsed['중도금지급일']))


## 5. 예제 추론 및 시스템 프롬프트 저장

먼저 `P3_qa.csv`의 예제 2개로 프롬프트를 확인합니다. 각 예제의 응답 원문과 형식 검사 결과가 셀 아래에 표시됩니다.\
프롬프트를 고쳤다면 함수 셀과 예제 추론 셀을 다시 실행하세요.

`gms_config.pkl`에는 호출에 사용할 OpenAI 객체가 준비되어 있습니다. 아래 코드는 `pickle.load()`로 표준 OpenAI 객체를 복원한 뒤 `client.chat.completions.create()`로 호출합니다. 연결 설정과 모델 정보는 객체에 저장되어 있으므로 별도로 입력하지 않습니다.

**연결 및 모델 정보를 임의로 바꾸거나 별도 주소로 우회를 시도하는 경우, 부정행위로 처리될 수 있습니다. 반드시 제공되는 모델만 사용해야 하며, 3번 문제 풀이 외의 목적으로 사용하는 경우에도 부정행위 처리될 수 있습니다.**

- 시스템 프롬프트는 `outputs/P3_<수험번호>.txt`에 UTF-8로 저장됩니다. 이 파일이 제출할 프롬프트 답안입니다.
- 예제 응답은 셀 출력에서 확인합니다. 형식 검사 통과가 내용의 정답을 뜻하지는 않습니다.
- 호출이나 형식 검사에 실패해도 다음 예제로 진행합니다. 다시 실행하면 현재 프롬프트로 예제를 재호출하고 TXT를 덮어씁니다.


In [ ]:
# 수정 금지

with Path('gms_config.pkl').open('rb') as source:
    client = pickle.load(source)
    client.api_key = USER_API_KEY

def call_llm(problem, system_prompt):
    if not isinstance(system_prompt, str) or not system_prompt.strip():
        raise ValueError('system_prompts()에서 비어 있지 않은 시스템 프롬프트 문자열을 반환하세요.')
    question = problem['question']
    if not isinstance(question, str) or not question.strip():
        raise ValueError('문항의 question은 비어 있지 않은 문자열이어야 합니다.')
    completion = client.chat.completions.create(
        model=client.model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': question},
        ],
    )
    if not completion.choices or completion.choices[0].finish_reason != 'stop':
        raise ValueError('모델 응답이 정상적으로 완료되지 않았습니다.')
    text = completion.choices[0].message.content
    if not isinstance(text, str) or not text.strip():
        raise ValueError('모델 응답이 비어 있습니다.')
    return text


In [ ]:
# 수정 금지

started = time.perf_counter()
system_prompt = system_prompts()
if not isinstance(system_prompt, str) or not system_prompt.strip():
    raise ValueError('system_prompts()에서 비어 있지 않은 시스템 프롬프트 문자열을 반환하세요.')

valid_count = 0
for index, problem in enumerate(examples):
    print('\n' + '=' * 60)
    print(f'예제 {index + 1}/{len(examples)} | {problem["task"]}')
    try:
        response_text = call_llm(problem, system_prompt)
    except Exception as error:
        print(f'호출 실패: {type(error).__name__}')
        continue

    print('모델 응답:')
    print(response_text)
    try:
        parse_response(response_text)
    except Exception as error:
        print(f'형식 검사 실패: {error}')
    else:
        valid_count += 1
        print('형식 검사 통과 (내용의 정답 여부를 의미하지 않습니다.)')

output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
output_path = output_dir / f'P3_{student_id}.txt'
with output_path.open('w', encoding='utf-8', newline='\n') as output:
    output.write(system_prompt)

print(f'\n시스템 프롬프트 저장: {output_path}')
print(f'예제 {len(examples)}개 / 형식 검사 통과 {valid_count}개')
print(f'Wall-clock: {time.perf_counter() - started:.1f}초')


## 제출

아래 파일들을 제출하세요.
- 완성한 노트북 `.ipynb`
- 생성된 `outputs/P3_<수험번호>.txt` 파일

파일명에는 첫 셀의 수험번호가 자동으로 들어갑니다. 따로 이름을 바꿀 필요가 없습니다.\
TXT에는 최종 시스템 프롬프트 문자열만 들어 있어야 합니다. 파일명에 맨 위 셀에서 입력한 수험번호가 사용되었는지 확인하세요.

프롬프트를 수정했다면 함수 셀과 예제 추론 셀을 다시 실행해 파일을 갱신하세요.\
예제 응답과 형식 검사 결과는 셀 출력에서 확인할 수 있습니다.
